# Predict and Verify Test Predictions (ViscosityPred)

This notebook reloads the GNN model weights stored in `ViscosityPred/` and re-runs
inference on each fold's test set. The recomputed predictions are compared with the
stored reference `predictions.csv` files to verify weight/code consistency.

**Requirements**: the `mixture_evaluation` GitHub repo must be available adjacent to
this `chemprop-1` repo (i.e. `../mixture_evaluation/` relative to the repo root).
This notebook uses the local `examples/ViscosityPred/train_gnn_ffn.py` together with
shared helper modules from `mixture_evaluation/scripts/scripts_training/utils/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "examples" / "ViscosityPred").exists():
            return p
    raise RuntimeError(f"Could not locate chemprop-1 repo root from {start}")


def _find_mix_eval_root(repo_root: Path) -> Path:
    candidates = [
        repo_root.parent / "mixture_evaluation",
        repo_root.parent.parent / "mixture_evaluation",
    ]
    for c in candidates:
        if (c / "scripts" / "scripts_training" / "utils" / "train_common.py").exists():
            return c
    raise RuntimeError(
        "Could not locate the mixture_evaluation repo. "
        "It must be a sibling of the chemprop-1 repo."
    )


repo_root = _find_repo_root(Path.cwd())
viscosity_dir = repo_root / "examples" / "ViscosityPred"
mix_eval_root = _find_mix_eval_root(repo_root)

# Path order matters here:
#   1. local ViscosityPred example dir for `import train_gnn_ffn`
#   2. current repo root for `model_core`
#   3. mixture_evaluation root for `scripts.scripts_training.utils.*`
for d in [str(viscosity_dir), str(repo_root), str(mix_eval_root)]:
    if d not in sys.path:
        sys.path.insert(0, d)

import train_gnn_ffn as tgm  # local ViscosityPred version
from scripts.scripts_training.utils.mixtures import collate_mixture
from model_core.featurizers import get_multi_hot_atom_featurizer
from model_core.featurizers.molgraph import SimpleMoleculeMolGraphFeaturizer

mix_csv = viscosity_dir / "viscosity_mix_exp.csv"
splits_dir = viscosity_dir / "mixture" / "cross_validation_5fold"
results_base = viscosity_dir / "viscosity_mix_exp" / "viscosity_mix_exp" / "mixture_combination"

print(f"repo_root      : {repo_root}")
print(f"viscosity_dir  : {viscosity_dir}")
print(f"mix_eval_root  : {mix_eval_root}")
print(f"mix_csv exists : {mix_csv.exists()}")
print(f"splits_dir     : {splits_dir}")
print(f"results_base   : {results_base}")

In [ ]:
# One job per cross-validation fold
jobs = [
    {
        "name": f"viscosity_mix_exp_mixture_combination_fold_{i:02d}",
        "fold_dir": results_base / f"fold_{i:02d}",
        "split_csv": splits_dir / f"fold_{i:02d}.csv",
    }
    for i in range(5)
]

output_dir = viscosity_dir / "repro_checks"
output_dir.mkdir(parents=True, exist_ok=True)
seed = 0
temperature_col = "temperature"
min_fraction = 1e-4

In [ ]:
def evaluate_viscosity_fold(
    fold_dir: Path,
    split_csv: Path,
    mix_csv: Path,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Reload model from *fold_dir* and rerun inference on the exact stored test split."""
    model_path = fold_dir / "model.pt"
    pred_csv_path = fold_dir / "predictions.csv"

    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    use_mixmp = ckpt.get("mixmp") is not None
    mixmp_type = "interaction" if use_mixmp else "none"
    temperature_mode = str(ckpt.get("temperature_mode", "none"))
    temperature_physics_law = str(ckpt.get("temperature_physics_law", "arrhenius"))
    atom_featurizer_mode = str(ckpt.get("atom_featurizer_mode", "ORGANIC"))

    stored = pd.read_csv(pred_csv_path, low_memory=False)
    y_true_ref = stored["y_true"].to_numpy(dtype=float)
    y_pred_stored = stored["y_pred"].to_numpy(dtype=float)

    train_df, val_df, test_df = tgm.load_split_dataframes(
        mix_csv=mix_csv,
        split_csv=split_csv,
        max_components=None,
        min_fraction=min_fraction,
        seed=seed,
    )
    df_mix = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)
    train_ids = list(range(0, len(train_df)))
    val_ids = list(range(len(train_df), len(train_df) + len(val_df)))
    test_ids = list(range(len(train_df) + len(val_df), len(df_mix)))

    if "solute_inchi" in df_mix.columns:
        solute_component_index = 0
    else:
        solute_component_index = -1

    if "_source_row_index" in test_df.columns:
        expected = stored["source_row_index"].astype(int).to_numpy()
        observed = test_df["_source_row_index"].astype(int).to_numpy()
        if not np.array_equal(expected, observed):
            raise AssertionError("Split/test row order does not match stored predictions.csv")

    temperature_features = tgm._build_temperature_features(
        df_mix,
        train_ids,
        temperature_col=temperature_col,
        temperature_mode=temperature_mode,
    )
    predictor_x_d_dim = 0 if temperature_features is None else int(temperature_features.shape[1])
    x_d_dim = predictor_x_d_dim

    if temperature_features is not None:
        all_data = tgm.build_all_data_with_xd(
            df_mix,
            x_d=temperature_features,
            target_col="value",
            solute_component_index=solute_component_index,
        )
    else:
        all_data = tgm.build_all_data(
            df_mix,
            target_col="value",
            solute_component_index=solute_component_index,
        )

    n_components = len(all_data) - 1
    train_data = tgm.take_rows(all_data, train_ids)
    test_data = tgm.take_rows(all_data, test_ids)

    component_featurizer = SimpleMoleculeMolGraphFeaturizer(
        atom_featurizer=get_multi_hot_atom_featurizer(atom_featurizer_mode)
    )
    train_mcdset = tgm.build_mixture_dataset(
        train_data,
        n_components,
        use_mixmp=use_mixmp,
        atom_featurizer_mode=atom_featurizer_mode,
        component_featurizer=component_featurizer,
    )
    scaler = train_mcdset.normalize_targets()
    test_mcdset = tgm.build_mixture_dataset(
        test_data,
        n_components,
        use_mixmp=use_mixmp,
        atom_featurizer_mode=atom_featurizer_mode,
        component_featurizer=component_featurizer,
    )
    test_mcdset.normalize_targets(scaler)
    loader = DataLoader(test_mcdset, batch_size=64, shuffle=False, collate_fn=collate_mixture)

    first_w = next(iter(ckpt["predictor"].values()))
    predictor_hidden_dim = int(first_w.shape[0])

    model = tgm.build_model(
        n_components=n_components,
        scaler=scaler,
        aggregation="weightedsum",
        solute_component_index=solute_component_index,
        mixmp_type=mixmp_type,
        x_d_dim=x_d_dim,
        temperature_mode=temperature_mode,
        temperature_physics_law=temperature_physics_law,
        atom_fdim=component_featurizer.atom_fdim,
        bond_fdim=component_featurizer.bond_fdim,
        predictor_hidden_dim=predictor_hidden_dim,
        predictor_x_d_dim=predictor_x_d_dim,
    )
    model.message_passing.load_state_dict(ckpt["message_passing"])
    if model.agg.mixmp is not None and ckpt.get("mixmp") is not None:
        model.agg.mixmp.load_state_dict(ckpt["mixmp"])
    model.agg.load_state_dict(ckpt["mixagg"], strict=False)
    model.predictor.load_state_dict(ckpt["predictor"])
    model.eval()

    preds = []
    with torch.no_grad():
        for batch in loader:
            bmgs, v_ds, x_d_batch, *_ = batch
            preds.append(model(bmgs, v_ds, x_d_batch).cpu().numpy().reshape(-1))
    y_pred_recomputed = np.concatenate(preds, axis=0)

    if len(y_pred_recomputed) != len(y_pred_stored):
        raise AssertionError(
            f"Prediction length mismatch: recomputed={len(y_pred_recomputed)} stored={len(y_pred_stored)}"
        )

    return y_true_ref, y_pred_recomputed, y_pred_stored

In [ ]:
rows = []
for job in jobs:
    name = job["name"]
    fold_dir = Path(job["fold_dir"])
    split_csv = Path(job["split_csv"])

    if not (fold_dir / "model.pt").exists():
        rows.append({"name": name, "status": "missing_model"})
        print(f"[SKIP] {name}: missing model.pt")
        continue
    if not (fold_dir / "predictions.csv").exists():
        rows.append({"name": name, "status": "missing_predictions"})
        print(f"[SKIP] {name}: missing predictions.csv")
        continue

    y_true, y_pred_new, y_pred_ref = evaluate_viscosity_fold(fold_dir, split_csv, mix_csv)
    n = min(len(y_pred_new), len(y_pred_ref))
    y_pred_new, y_pred_ref, y_true = y_pred_new[:n], y_pred_ref[:n], y_true[:n]
    diff = y_pred_new - y_pred_ref

    rows.append(
        {
            "name": name,
            "status": "ok",
            "n_compared": int(n),
            "exact_match": bool(np.array_equal(y_pred_new, y_pred_ref)),
            "allclose_atol_1e6": bool(np.allclose(y_pred_new, y_pred_ref, atol=1e-6, rtol=0.0)),
            "max_abs_diff": float(np.max(np.abs(diff))) if n > 0 else float("nan"),
        }
    )

    pd.DataFrame(
        {
            "y_true": y_true,
            "y_pred_recomputed": y_pred_new,
            "y_pred_reference": y_pred_ref,
            "delta": diff,
        }
    ).to_csv(output_dir / f"{name}_prediction_comparison.csv", index=False)

    print(f"[OK] {name}: n={n}, max_abs_diff={float(np.max(np.abs(diff))):.3e}")

summary = pd.DataFrame(rows)
summary.to_csv(output_dir / "mole_gnn_viscosity_test_repro_summary.csv", index=False)
display(summary)